# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all available record sets and their @id
print('Available Record Sets:')
record_sets = list(metadata.record_sets)
for idx, rs in enumerate(record_sets):
    print(f"{idx+1}. ID: {rs['@id']} | name: {rs.get('name', 'N/A')}")

# For each record set, list its fields and columns referenced by @id
for rs in record_sets:
    print(f"\nRecordSet: {rs['@id']} - {rs.get('name','N/A')}")
    print('  Fields:')
    for fld in rs.get('field', []):
        field_obj = metadata.get_by_id(fld)
        print(f"    - Field @id: {field_obj['@id']} | name: {field_obj.get('name', 'N/A')} | dataType: {field_obj.get('dataType', 'N/A')}")
        # List columns for this field
        if 'column' in field_obj:
            # The column value(s) might be a list or a string
            columns = field_obj['column'] if isinstance(field_obj['column'], list) else [field_obj['column']]
            for col in columns:
                col_obj = metadata.get_by_id(col)
                print(f"      -> Column @id: {col_obj['@id']} | name: {col_obj.get('name','N/A')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Prepare to extract data from each record set
record_set_ids = [rs['@id'] for rs in metadata.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading records for RecordSet: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records. Columns: {df.columns.tolist()}")
    else:
        print(f"No records found for {record_set_id}.")

# As an example, select the first record set for demonstration; adjust as needed
if dataframes:
    example_record_set_id = list(dataframes.keys())[0]
    print(f"\nFirst few rows from RecordSet {example_record_set_id}:")
    display(dataframes[example_record_set_id].head())
else:
    example_record_set_id = None
    print("No record sets with data found.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Automated field selection for demonstration
import numpy as np
if example_record_set_id is not None:
    df = dataframes[example_record_set_id]
    # Try to pick a numeric field by searching record set fields with Integer/Float type
    record_set_obj = metadata.get_by_id(example_record_set_id)
    numeric_field_id = None
    group_field_id = None
    for fld in record_set_obj.get('field', []):
        field_obj = metadata.get_by_id(fld)
        dt = str(field_obj.get('dataType',''))
        # crude heuristic for numeric
        if 'Integer' in dt or 'Float' in dt or 'Number' in dt:
            potential_field_id = field_obj['@id']
            # Use field name as column if present
            field_name = field_obj.get('name', potential_field_id)
            if field_name in df.columns and np.issubdtype(df[field_name].dtype, np.number):
                numeric_field_id = field_name
                break
    # Try to choose a group field as the first non-numeric
    for fld in record_set_obj.get('field', []):
        field_obj = metadata.get_by_id(fld)
        field_name = field_obj.get('name', field_obj['@id'])
        if field_name != numeric_field_id and field_name in df.columns and df[field_name].dtype == object:
            group_field_id = field_name
            break

    print(f"Numeric field for analysis: {numeric_field_id}")
    print(f"Group field for categorical analysis: {group_field_id}")

    # Ensure numeric, drop non-numeric entries
    df_num = df.copy()
    if numeric_field_id is not None:
        df_num = df_num[pd.to_numeric(df_num[numeric_field_id], errors='coerce').notnull()]
        df_num[numeric_field_id] = pd.to_numeric(df_num[numeric_field_id])
        threshold = df_num[numeric_field_id].mean()+df_num[numeric_field_id].std()  # Use mean+std for demonstration
        filtered_df = df_num[df_num[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        display(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped data by {group_field_id} (mean {numeric_field_id}):")
            display(grouped_df.head())
        else:
            print("No suitable group field for this dataset.")
    else:
        print("No numeric field found for EDA.")
else:
    print("No record set with data to analyze.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if example_record_set_id is not None and numeric_field_id is not None:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=30)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    if group_field_id is not None:
        # Show boxplot by group
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df.dropna(subset=[numeric_field_id, group_field_id]))
        plt.title(f'{numeric_field_id} by {group_field_id}')
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to:
- Load a Croissant-structured FAIR dataset using `mlcroissant`
- Examine the hierarchical structure of record sets, fields, and columns using their `@id`
- Extract structured tabular data, perform exploratory filtering, normalization, and grouping
- Visualize distributions and relationships for key features

Further steps could include building machine learning pipelines for protein abundance or modification prediction, performing advanced clustering analysis, or integrating domain-specific bioinformatics tools.

Explore additional record sets and use the `@id` fields for consistent, machine-actionable, and reproducible FAIR data science!